# Find THEMIS IRPBT products over Gale Crater (ODE REST API)

Searches the Mars Orbital Data Explorer (ODE) for THEMIS projected brightness
temperature products (IRPBT) whose footprints fall in a box around Gale Crater,
then saves the product list to CSV.

**Status:** written from the ODE REST docs and examples, not yet run against the
live server. Each step prints its raw response so you can see what ODE returned.
If a parameter is rejected, check the manual:
https://oderest.rsl.wustl.edu/ODE_REST_V2.1.6.pdf

ODE longitudes are 0-360 east.

## 1. Setup and search box

In [1]:
import re
import requests
import pandas as pd

BASE = "https://oderest.rsl.wustl.edu/live2/"

# Gale center is about 5.4 S, 137.8 E; the crater is ~154 km wide (~2.6 deg).
# This box adds a margin. Edit to taste.
BBOX = dict(minlat=-6.57, maxlat=-4.15, westernlon=136.75, easternlon=139.24)

PAGE = 500  # products per request


def ode(params):
    """Call the ODE REST endpoint and return parsed JSON."""
    r = requests.get(BASE, params={**params, "output": "JSON"}, timeout=120)
    r.raise_for_status()
    return r.json()


def find_values(obj, pattern):
    """Recursively collect string values matching a regex anywhere in the JSON."""
    found = set()
    if isinstance(obj, dict):
        for v in obj.values():
            found |= find_values(v, pattern)
    elif isinstance(obj, list):
        for v in obj:
            found |= find_values(v, pattern)
    elif isinstance(obj, str) and re.fullmatch(pattern, obj):
        found.add(obj)
    return found

## 2. Find the exact IRPBT product type names

ODE often appends a version to the type name (for example `IRPBTV10` or
`IRPBTV20`), so ask ODE instead of assuming the bare name `IRPBT`.

In [2]:
iipt = ode({"query": "iipt", "target": "mars"})
ptypes = sorted(find_values(iipt, r"IRPBT\w*"))
print("IRPBT product types found:", ptypes)

# If this is empty, inspect `iipt` and set ptypes by hand, e.g.:
# ptypes = ["IRPBTV20"]
assert ptypes, "No IRPBT types found; inspect the iipt response above."

IRPBT product types found: ['IRPBT1', 'IRPBT2', 'IRPBT4']


## 3. Count matching products

Check the scale before pulling metadata.

In [3]:
def base_params(pt):
    return {
        "query": "product",
        "target": "mars",
        "ihid": "ODY",
        "iid": "THEMIS",
        "pt": pt,
        **BBOX,
    }


for pt in ptypes:
    cnt = ode({**base_params(pt), "results": "c"})
    print(pt, "->", cnt.get("ODEResults", {}).get("Count", cnt))

IRPBT1 -> 146
IRPBT2 -> 380
IRPBT4 -> 276


## 4. Pull metadata for every match

Pages through the results (`results=m` is the metadata result code; confirm in
the manual if ODE rejects it).

In [4]:
rows = []
for pt in ptypes:
    offset = 0
    while True:
        res = ode({**base_params(pt), "results": "m", "limit": PAGE, "offset": offset})
        prods = res.get("ODEResults", {}).get("Products", {}).get("Product", [])
        if isinstance(prods, dict):  # a single hit comes back as a dict
            prods = [prods]
        if not prods:
            break
        for p in prods:
            p["queried_pt"] = pt
        rows.extend(prods)
        if len(prods) < PAGE:
            break
        offset += PAGE

df = pd.json_normalize(rows)
print(f"{len(df)} products")
df.head()

802 products


,BB_georeferenced,Center_georeferenced,Center_latitude,Center_longitude,Description,Easternmost_longitude,Emission_angle,External_url,FilesURL,Footprint_C0_geometry,...,Start_orbit_number,Stop_orbit_number,UTC_start_time,UTC_stop_time,Westernmost_longitude,queried_pt,ODE_notes.ODE_note,USGS_Sites.USGS_Site,Product_name,Solar_distance
0,True,True,-4.478,137.627,-5 deg night atmos,137.922,0.569,https://viewer.mars.asu.edu/planetview/inst/th...,https://ode.rsl.wustl.edu/mars/productfiles.as...,"POLYGON ((137.86 -4.236, 137.922 -4.692, 137.3...",...,3160,3160,2002-08-31T17:08:25.861,2002-08-31T17:08:34.924,137.331,IRPBT1,"[NOTE: Product Type set by ODE, NOTE: Instrume...",Gale,NaN,NaN
1,True,True,-7.05,138.698,Eroded crater wall - MSIP,139.374,1.519,https://viewer.mars.asu.edu/planetview/inst/th...,https://ode.rsl.wustl.edu/mars/productfiles.as...,"POLYGON ((138.02 -10.067, 138.844 -4.008, 139....",...,4415,4415,2002-12-12T23:30:23.684,2002-12-12T23:32:23.683,138.02,IRPBT1,"[NOTE: Product Type set by ODE, NOTE: Instrume...",NaN,NaN,NaN
2,True,True,-5.542,136.913,Gale Crater,137.995,1.537,https://viewer.mars.asu.edu/planetview/inst/th...,https://ode.rsl.wustl.edu/mars/productfiles.as...,"POLYGON ((135.822 -11.575, 137.462 0.507, 137....",...,4851,4851,2003-01-17T22:46:39.131,2003-01-17T22:50:38.603,135.822,IRPBT1,"[NOTE: Product Type set by ODE, NOTE: Instrume...",NaN,NaN,NaN
3,True,True,-3.017,139.009,Systematic Mapping,140.092,0.543,https://viewer.mars.asu.edu/planetview/inst/th...,https://ode.rsl.wustl.edu/mars/productfiles.as...,"POLYGON ((138.462 3.031, 140.092 -9.047, 139.5...",...,5020,5020,2003-01-31T19:35:55.324,2003-01-31T19:39:54.796,137.926,IRPBT1,"[NOTE: Product Type set by ODE, NOTE: Instrume...",NaN,NaN,NaN
4,True,True,-3.962,136.908,Crater fill material,137.38,1.405,https://viewer.mars.asu.edu/planetview/inst/th...,https://ode.rsl.wustl.edu/mars/productfiles.as...,"POLYGON ((136.437 -5.466, 136.848 -2.43, 137.3...",...,5601,5601,2003-03-20T14:30:36.933,2003-03-20T14:31:37.198,136.437,IRPBT1,"[NOTE: Product Type set by ODE, NOTE: Instrume...",NaN,NaN,NaN


## 5. Inspect columns and save

Confirm the table has local solar time and solar longitude (Ls), since matching
to your model slices depends on them. If not, read them from each product's PDS
label. Also check whether ODE's default location match is "overlaps the box" or
"fully inside the box" in the manual.

In [5]:
print(list(df.columns))
df.to_csv("gale_themis_irpbt.csv", index=False)
print("Saved gale_themis_irpbt.csv")

['BB_georeferenced', 'Center_georeferenced', 'Center_latitude', 'Center_longitude', 'Description', 'Easternmost_longitude', 'Emission_angle', 'External_url', 'FilesURL', 'Footprint_C0_geometry', 'Footprint_geometry', 'Footprint_GL_geometry', 'Footprint_NP_geometry', 'Footprint_souce', 'Footprint_SP_geometry', 'Footprints_cross_meridian', 'Incidence_angle', 'LabelURL', 'Map_resolution', 'Map_scale', 'Maximum_latitude', 'Minimum_latitude', 'Observation_time', 'Phase_angle', 'Pole_state', 'Producer_id', 'ProductURL', 'Solar_longitude', 'Solar_time', 'SpaceCraft_clock_start_count', 'SpaceCraft_clock_stop_count', 'Start_orbit_number', 'Stop_orbit_number', 'UTC_start_time', 'UTC_stop_time', 'Westernmost_longitude', 'queried_pt', 'ODE_notes.ODE_note', 'USGS_Sites.USGS_Site', 'Product_name', 'Solar_distance']
Saved gale_themis_irpbt.csv


## 6. Clean the table and score footprint coverage

Steps 1-5 only fetch the catalog (one row per product). This step prepares that
table for filtering. It reloads the CSV, so you can restart here without
re-querying ODE.

What it fixes or adds:
- `product_id`: `Product_name` is empty, so the ID is pulled from `ProductURL`.
- `ltst_h`: `Solar_time` mixes decimal hours (`4.309`) and `HH:MM:SS` strings
  (`05:28:58.8`), so both are converted to decimal hours of local solar time.
- `m_per_px`: meters per pixel. ODE's `Map_resolution` and `Map_scale` columns swap
  meaning between product types (IRPBT1/2 store pixels per degree in
  `Map_resolution` and meters per pixel in `Map_scale`; IRPBT4 is the reverse), so
  never compare them directly across types.
- `gale_cover`: the fraction of Gale (a ~77 km radius disk) inside each footprint.
- `covers_point`: whether the footprint contains `TARGET`.

ODE returns anything that *overlaps* the search box, so many rows only clip a
corner of it. Each THEMIS image is a narrow strip, far smaller than the crater,
so expect low `gale_cover` for single images. A crater-wide map needs several
strips mosaicked at similar local time and Ls.

In [6]:
import re
import numpy as np
import pandas as pd

df = pd.read_csv("gale_themis_irpbt.csv", low_memory=False)

# --- targets (edit) ---------------------------------------------------------
CRATER = dict(lon=137.8, lat=-5.4, radius_deg=1.3)   # ~77 km radius near the equator
TARGET = (137.4417, -4.5895)                         # (lon E, lat) Bradbury Landing;
                                                     # swap in a rover position later


def to_hours(v):
    """'4.309' or '05:28:58.8' -> decimal hours."""
    s = str(v).strip()
    if ":" in s:
        h, m, sec = (s.split(":") + ["0", "0"])[:3]
        return int(h) + int(m) / 60 + float(sec) / 3600
    return float(s)


def parse_polygon(wkt):
    """'POLYGON ((lon lat, lon lat, ...))' -> (n, 2) array of lon, lat."""
    return np.array([[float(x) for x in p.split()]
                     for p in re.findall(r"(-?[\d.]+ -?[\d.]+)", wkt)])


def inside(poly, px, py):
    """Ray casting: which of the points (px, py) fall inside the polygon ring."""
    x, y = poly[:, 0], poly[:, 1]
    res = np.zeros(px.shape, bool)
    j = len(poly) - 1
    for i in range(len(poly)):
        res ^= ((y[i] > py) != (y[j] > py)) & (
            px < (x[j] - x[i]) * (py - y[i]) / (y[j] - y[i] + 1e-12) + x[i])
        j = i
    return res


# grid of sample points covering the crater disk
r = CRATER["radius_deg"]
g = np.linspace(-r, r, 41)
gx, gy = np.meshgrid(CRATER["lon"] + g, CRATER["lat"] + g)
disk = (gx - CRATER["lon"]) ** 2 + (gy - CRATER["lat"]) ** 2 <= r ** 2
px, py = gx[disk], gy[disk]

polys = df.Footprint_geometry.map(parse_polygon)
df["product_id"] = df.ProductURL.str.extract(r"product_id=([^&]+)")
df["ltst_h"] = df.Solar_time.map(to_hours)
# IRPBT4 lists meters/pixel in Map_resolution; IRPBT1 and IRPBT2 list it in Map_scale
df["m_per_px"] = np.where(df.queried_pt == "IRPBT4", df.Map_resolution, df.Map_scale)

crater_masks = polys.map(lambda p: inside(p, px, py))       # per-product covered points
df["gale_cover"] = crater_masks.map(np.mean)
df["covers_point"] = polys.map(
    lambda p: bool(inside(p, np.array([TARGET[0]]), np.array([TARGET[1]]))[0]))

print(len(df), "products;", df.product_id.duplicated().sum(), "duplicate IDs")
print("best single-image crater coverage:", round(df.gale_cover.max(), 2))
print("products covering TARGET:", int(df.covers_point.sum()))
print(df.groupby("queried_pt").m_per_px.agg(["min", "max"]).to_string())
df[["product_id", "queried_pt", "ltst_h", "Solar_longitude", "m_per_px",
    "gale_cover", "covers_point"]].head()

802 products; 0 duplicate IDs
best single-image crater coverage: 0.84
products covering TARGET: 184
              min    max
queried_pt              
IRPBT1      100.0  100.0
IRPBT2      100.0  200.0
IRPBT4      100.0  400.0


,product_id,queried_pt,ltst_h,Solar_longitude,m_per_px,gale_cover,covers_point
0,I03160022PBT,IRPBT1,4.309,62.152,100.0,0.046363,True
1,I04415001PBT,IRPBT1,16.743,107.716,100.0,0.116707,False
2,I04851011PBT,IRPBT1,16.894,124.381,100.0,0.191847,False
3,I05020011PBT,IRPBT1,4.984,131.008,100.0,0.002398,False
4,I05601001PBT,IRPBT1,17.167,154.936,100.0,0.086331,False


## 7. Filter to the observations you need

Set your windows below. Defaults keep every Ls and every local time that actually
exists in the data, so first look at the coverage table in step 8, then narrow.

- **Point mode** (`MODE = "point"`): products whose footprint contains `TARGET`.
  Use this for matching REMS at the rover.
- **Crater mode** (`MODE = "crater"`): products covering at least `MIN_COVER` of
  the crater, for spatial validation across Gale.

Observations are de-duplicated on observation time, keeping the finest
`m_per_px` (smallest meters per pixel), in case the same observation appears in
more than one product type (IRPBT1, IRPBT2, IRPBT4).

In [7]:
MODE = "crater"                       # "point" or "crater"
MIN_COVER = 0.25                     # used in crater mode
LS_RANGE = (0, 360)                  # solar longitude, degrees
LTST_WINDOWS = [(0, 24)]             # local solar time, hours; e.g. [(3, 7), (15, 19)]

if MODE == "point":
    where = df.covers_point
else:
    where = df.gale_cover >= MIN_COVER

in_ltst = np.zeros(len(df), bool)
for lo, hi in LTST_WINDOWS:
    in_ltst |= ((df.ltst_h >= lo) & (df.ltst_h < hi)).to_numpy()

sel = df[where & in_ltst & df.Solar_longitude.between(*LS_RANGE)].copy()

# same observation time -> keep the finest resolution
sel["obs_key"] = sel.Observation_time.str[:19]
sel = (sel.sort_values("m_per_px")
          .drop_duplicates("obs_key", keep="first")
          .sort_values("Observation_time"))

print(len(sel), "products selected")
print(sel.queried_pt.value_counts().to_dict())

# how much of the crater do the selected strips cover together?
if len(sel):
    union = np.zeros(px.shape, bool)
    for i in sel.index:
        union |= crater_masks[i]
    print("selected products jointly cover", round(union.mean() * 100), "% of the crater")

sel[["product_id", "queried_pt", "Observation_time", "ltst_h", "Solar_longitude",
     "m_per_px", "gale_cover"]]

#Filter to gale_cover > 30%
sel = sel[sel['gale_cover'] > 0.3]

sel.head()

225 products selected
{'IRPBT2': 108, 'IRPBT4': 86, 'IRPBT1': 31}
selected products jointly cover 100 % of the crater


,BB_georeferenced,Center_georeferenced,Center_latitude,Center_longitude,Description,Easternmost_longitude,Emission_angle,External_url,FilesURL,Footprint_C0_geometry,...,ODE_notes.ODE_note,USGS_Sites.USGS_Site,Product_name,Solar_distance,product_id,ltst_h,m_per_px,gale_cover,covers_point,obs_key
31,True,True,-5.325,138.041,No Description Given,138.576,28.343,https://viewer.mars.asu.edu/planetview/inst/th...,https://ode.rsl.wustl.edu/mars/productfiles.as...,"POLYGON ((137.49 -6.825, 137.898 -3.795, 138.5...",...,"['NOTE: Product Type set by ODE', 'NOTE: Instr...",NaN,NaN,NaN,I34540001PBT,14.737,100.0,0.322942,False,2009-09-27T07:24:48
33,True,True,-4.712,137.336,No Description Given,138.088,28.250,https://viewer.mars.asu.edu/planetview/inst/th...,https://ode.rsl.wustl.edu/mars/productfiles.as...,"POLYGON ((137.279 -1.666, 138.088 -7.719, 137....",...,"['NOTE: Product Type set by ODE', 'NOTE: Instr...",Gale,NaN,NaN,I34571007PBT,3.210,100.0,0.311751,True,2009-09-29T21:35:19
36,True,True,-4.872,137.605,No Description Given,138.445,28.261,https://viewer.mars.asu.edu/planetview/inst/th...,https://ode.rsl.wustl.edu/mars/productfiles.as...,"POLYGON ((137.468 -1.183, 138.445 -8.523, 137....",...,"['NOTE: Product Type set by ODE', 'NOTE: Instr...",Gale,NaN,NaN,I34883005PBT,3.315,100.0,0.322942,True,2009-10-25T14:04:02
38,True,True,-4.434,137.514,No Description Given,138.652,22.564,https://viewer.mars.asu.edu/planetview/inst/th...,https://ode.rsl.wustl.edu/mars/productfiles.as...,"POLYGON ((137.044 1.613, 138.652 -10.451, 138....",...,"['NOTE: Product Type set by ODE', 'NOTE: Instr...",Gale,NaN,NaN,I35195003PBT,3.404,100.0,0.300560,True,2009-11-20T06:31:11
680,True,True,-1.281,137.141,"MARS ODYSSEY THEMIS Projected Data Products, I...",138.413,1.674,https://viewer.mars.asu.edu/viewer/themis/i886...,https://ode.rsl.wustl.edu/mars/productfiles.as...,"POLYGON ((135.869 -8.84, 135.869 -7.84, 135.86...",...,['NOTE: ODE Changed Product Type DDR to IRPBT4...,Gale,I88655002PBT.IMG,1.565,i88655002pbt,7.077,100.0,0.790568,True,2021-12-09T03:46:41


## 8. Coverage table: local time vs season

Counts of selected products by 2-hour local-solar-time bin and 90-degree Ls bin.
Use this to see which parts of the diurnal cycle orbital data can validate before
you download anything. Empty rows mean no observations at those local times.

In [8]:
cov = pd.crosstab(
    (sel.ltst_h // 2 * 2).astype(int).rename("LTST bin (h, start)"),
    (sel.Solar_longitude // 90 * 90).astype(int).rename("Ls bin (deg, start)"),
)
cov = cov.reindex(range(0, 24, 2), fill_value=0)
cov

"Ls bin (deg, start)",0,90,180,270
"LTST bin (h, start)",,,,
0,0,0,0,0
2,1,0,0,2
4,0,0,0,0
6,0,5,5,0
8,0,0,0,0
10,0,0,0,0
12,0,0,0,0
14,0,0,0,1
16,0,0,0,0


## 9. Download the selected products

What `LabelURL` points to depends on the product type:

- **IRPBT1 and IRPBT2:** an `.IMG`, which is the image itself (an ASCII PDS3 label
  followed by binary pixels). Despite the extension it is not a disc image, and
  Windows may label it "Disc Image File".
- **IRPBT4:** an `.xml` PDS4 label only, with no pixels in it. Windows may show it as
  an HTML document. The image is the `.IMG` with the same name in the same folder
  (the label's `<file_name>`). This cell downloads both.

Files are served from `static.mars.asu.edu`. Sizes seen so far: about 3 MB per IRPBT1
`.IMG` and about 11 MB per IRPBT4 `.IMG`; IRPBT2 sizes are unchecked, so keep
`DRY_RUN` on until you have checked the count. The ASU host refused my own automated
fetch, so if scripted downloads fail for you too, open the product pages in a
browser instead (`FilesURL` links to ODE's file list for each product).

In [9]:
import pathlib
import time
import requests

DRY_RUN = False                        # set False to actually download
OUT = pathlib.Path("themis_irpbt")
DELAY_S = 1.0                         # be polite to the server

OUT.mkdir(exist_ok=True)

# IRPBT4 rows list the PDS4 .xml label, not the image; the image is the .IMG with
# the same name in the same folder. Download the image for every product, plus the
# .xml label where there is one (it carries the data type and map geometry).
sel["image_url"] = sel.LabelURL.str.replace(".xml", ".IMG", regex=False)
sel["label_url"] = sel.LabelURL.where(sel.LabelURL.str.endswith(".xml"))

urls = list(sel.image_url) + list(sel.label_url.dropna())
print(len(sel), "products,", len(urls), "files would be downloaded to", OUT.resolve())

if not DRY_RUN:
    for url in urls:
        dest = OUT / url.rsplit("/", 1)[-1]
        if dest.exists():
            continue
        try:
            r = requests.get(url, timeout=300)
            r.raise_for_status()
        except requests.RequestException as e:
            print("FAILED", url, e)
            continue
        dest.write_bytes(r.content)
        print("saved", dest.name, f"{len(r.content) / 1e6:.1f} MB")
        time.sleep(DELAY_S)

sel.to_csv("gale_themis_irpbt_selected.csv", index=False)
print("Saved gale_themis_irpbt_selected.csv")

23 products, 42 files would be downloaded to C:\Users\aypen\Downloads\MarsDataPipeline\themis_irpbt
Saved gale_themis_irpbt_selected.csv


## 10. Read the images and sample brightness temperature at the target

The two formats store pixels differently, so the reader handles both:

- **IRPBT1 (PDS3 `.IMG`, checked on `I34571007PBT`):** 8-bit integers. Kelvin is
  `DN * SCALING_FACTOR + OFFSET`, and DN 0 means no data. Each image has its own
  scale and offset, so raw DNs cannot be compared between images.
- **IRPBT4 (PDS4 `.xml` + `.IMG`, from the `IA4884001PBT` label):** 32-bit
  little-endian floats already in kelvin, with 0 meaning no data. Not yet tested on
  a real IRPBT4 image: I only had its label.
- **IRPBT2:** not checked. The reader uses the attached PDS3 label and raises an
  error naming any missing keyword if it differs.

Both are sinusoidal maps, so `to_rowcol` converts longitude and latitude to a pixel.
Strips are tilted inside their rectangular frame, so many pixels are empty and a
location can fall in the frame yet have no data. The function returns NaN then.

For IRPBT4 the label also carries `GEOMETRY_QUALITY_RATING` and
`ATTITUDE_QUALITY_RATING`; the example had `NO-ISSUES` and `CHECK-STABILITY`.

In [10]:
import os
import xml.etree.ElementTree as ET

R_MARS = 3396190.0  # m, the sphere both products use

PDS4_TYPES = {"IEEE754LSBSingle": "<f4", "IEEE754MSBSingle": ">f4", "UnsignedByte": "u1"}


def _num(s):
    return float(str(s).split()[0])           # drops a trailing "<UNIT>"


def read_pds3_label(path, max_bytes=65536):
    """Attached PDS3 label at the top of an .IMG -> dict of KEY -> value string."""
    with open(path, "rb") as f:
        text = f.read(max_bytes).decode("latin-1")
    label = {}
    for line in text.splitlines():
        if line.strip() == "END":
            break
        if "=" in line:
            k, v = line.split("=", 1)
            label.setdefault(k.strip(), v.strip().strip('"'))
    return label


def read_pds3_image(path):
    """PDS3 .IMG -> (kelvin float array with NaN for no data, to_rowcol, info)."""
    h = read_pds3_label(path)
    for key in ("LINES", "LINE_SAMPLES", "SAMPLE_BITS", "SAMPLE_TYPE"):
        if key not in h:
            raise KeyError(f"{path}: label has no {key}; check its format by hand")
    lines, samples = int(h["LINES"]), int(h["LINE_SAMPLES"])
    stype = h["SAMPLE_TYPE"].upper()
    kind = "f" if "REAL" in stype else "u"
    endian = "<" if stype.startswith(("PC_", "LSB")) else ">"
    dtype = np.dtype(endian + kind + str(int(h["SAMPLE_BITS"]) // 8))
    offset = os.path.getsize(path) - lines * samples * dtype.itemsize  # image is last
    if offset < 0:
        raise ValueError(f"{path}: file is smaller than {lines}x{samples} {dtype}")
    arr = np.fromfile(path, dtype=dtype, offset=offset, count=lines * samples)
    arr = arr.reshape(lines, samples)
    K = arr.astype("float64") * _num(h.get("SCALING_FACTOR", 1)) + _num(h.get("OFFSET", 0))
    K[arr == _num(h.get("NULL_CONSTANT", 0))] = np.nan
    K[~np.isfinite(K)] = np.nan

    ppd, lat_top = _num(h["MAP_RESOLUTION"]), _num(h["MAXIMUM_LATITUDE"])
    lon0, soff = _num(h["CENTER_LONGITUDE"]), -_num(h["SAMPLE_PROJECTION_OFFSET"])

    def to_rowcol(lon, lat):                    # sinusoidal, lon in 0-360 east
        return ((lat_top - lat) * ppd,
                (lon - lon0) * np.cos(np.radians(lat)) * ppd + soff)

    return K.astype("float32"), to_rowcol, {"format": "PDS3", "quality": None}


def read_pds4_image(xml_path):
    """PDS4 .xml label + sibling .IMG -> (kelvin array, to_rowcol, info)."""
    root = ET.parse(xml_path).getroot()
    vals, params, axes = {}, {}, {}
    for el in root.iter():
        tag = el.tag.split("}")[-1]
        kids = {k.tag.split("}")[-1]: (k.text or "").strip() for k in el}
        if tag == "Parameter":
            params[kids.get("name")] = kids.get("value")
        elif tag == "Axis_Array":
            axes[kids["axis_name"]] = int(kids["elements"])
        elif el.text and el.text.strip():
            vals.setdefault(tag, el.text.strip())
    img = os.path.splitext(xml_path)[0] + ".IMG"
    if not os.path.exists(img):
        img = os.path.splitext(xml_path)[0] + ".img"
    dtype = np.dtype(PDS4_TYPES[vals["data_type"]])
    lines, samples = axes["Line"], axes["Sample"]
    offset = os.path.getsize(img) - lines * samples * dtype.itemsize
    arr = np.fromfile(img, dtype=dtype, offset=offset, count=lines * samples)
    K = arr.reshape(lines, samples).astype("float64")
    K[K == float(vals.get("missing_constant", 0))] = np.nan
    K[~np.isfinite(K)] = np.nan

    lon0 = float(vals["longitude_of_central_meridian"])
    ulx, uly = float(vals["upperleft_corner_x"]), float(vals["upperleft_corner_y"])
    res = float(vals["pixel_resolution_x"])

    def to_rowcol(lon, lat):
        x = R_MARS * np.radians(lon - lon0) * np.cos(np.radians(lat))
        y = R_MARS * np.radians(lat)
        return (uly - y) / res, (x - ulx) / res

    quality = {k: params.get(k) for k in ("GEOMETRY_QUALITY_RATING", "ATTITUDE_QUALITY_RATING")}
    return K.astype("float32"), to_rowcol, {"format": "PDS4", "quality": quality}


def read_product(image_path):
    """Use the PDS4 label if one sits next to the image, otherwise the PDS3 header."""
    xml = os.path.splitext(str(image_path))[0] + ".xml"
    return read_pds4_image(xml) if os.path.exists(xml) else read_pds3_image(str(image_path))


def sample_at(K, to_rowcol, lon, lat, half=2):
    """Mean of valid pixels in a (2*half+1)^2 window around lon, lat; NaN if none."""
    row, col = to_rowcol(lon, lat)
    r, c = int(row), int(col)
    if not (0 <= r < K.shape[0] and 0 <= c < K.shape[1]):
        return np.nan
    win = K[max(r - half, 0):r + half + 1, max(c - half, 0):c + half + 1]
    return float(np.nanmean(win)) if np.isfinite(win).any() else np.nan


rows = []
missing = 0
for _, p in sel.iterrows():                     # run step 9 first (it defines image_url)
    path = OUT / p.image_url.rsplit("/", 1)[-1]
    if not path.exists():
        missing += 1
        continue
    K, to_rowcol, info = read_product(path)
    rows.append(dict(
        product_id=p.product_id, queried_pt=p.queried_pt,
        Observation_time=p.Observation_time, ltst_h=p.ltst_h,
        Solar_longitude=p.Solar_longitude, m_per_px=p.m_per_px,
        T_target_K=sample_at(K, to_rowcol, *TARGET),
        valid_frac=float(np.isfinite(K).mean()),
        **(info["quality"] or {}),
    ))

temps = pd.DataFrame(rows)
print(len(temps), "products read;", missing, "not downloaded yet")
if len(temps):
    temps.to_csv("themis_target_temps.csv", index=False)
    print("Saved themis_target_temps.csv")
temps.head()

23 products read; 0 not downloaded yet
Saved themis_target_temps.csv


,product_id,queried_pt,Observation_time,ltst_h,Solar_longitude,m_per_px,T_target_K,valid_frac,GEOMETRY_QUALITY_RATING,ATTITUDE_QUALITY_RATING
0,I34540001PBT,IRPBT1,2009-09-27T07:24:48.010,14.737,345.002,100.0,NaN,0.602183,NaN,NaN
1,I34571007PBT,IRPBT1,2009-09-29T21:35:19.830,3.210,346.357,100.0,207.352951,0.447225,NaN,NaN
2,I34883005PBT,IRPBT1,2009-10-25T14:04:02.089,3.315,359.476,100.0,201.964935,0.403912,NaN,NaN
3,I35195003PBT,IRPBT1,2009-11-20T06:31:11.313,3.404,12.071,100.0,199.123871,0.277735,NaN,NaN
4,i88655002pbt,IRPBT4,2021-12-09T03:46:41.955Z,7.077,139.147,100.0,197.764603,0.216191,CAUTION,UNSTABLE


## 11. Next steps

- Compare `T_target_K` to REMS ground temperature at matching local time (within ~30 min)
  and Ls (within a few degrees). Drop rows where `T_target_K` is NaN (the target fell
  in an empty part of the strip).
- Report bias and RMSE by LTST bin, and report orbital-vs-REMS measurement bias separately.
- Confirm what IRPBT1, IRPBT2 and IRPBT4 are in ODE before mixing them in one analysis.
- For IRPBT4, consider dropping products whose `GEOMETRY_QUALITY_RATING` is not `NO-ISSUES`.